# **TP2 — PARCOURS DATA SCIENCE: Segmentation Clients RFM**

**CONTEXTE DU PROJET**

Un e-commerce britannique de cadeaux, vendant surtout en gros (« wholesale »), a accumulé plus d'un million de
transactions sur deux ans. La direction marketing veut arrêter de tirer à l'aveugle : elle veut savoir qui sont ses clients
pour choyer les meilleurs, relancer ceux qui s'éloignent et réveiller ceux qui dorment.

**TRAVAIL A FAIRE**

 construire une segmentation clients avec la méthode RFM (Récence, Fréquence, Montant), choisir un
nombre de segments k justifié, nommer et caractériser ces segments, puis formuler des recommandations marketing
actionnables. La méthode transpose directement à un e-commerce ouest-africain (ex. plateforme de paiement mobile,
marketplace locale)

**0. Etape préliminaire: Importation des bibliothèques et de la base de données**

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set()

In [27]:
#Chargement et aperçu de la base de données
df_2009 = pd.read_excel("/content/online_retail_II.xlsx", sheet_name="Year 2009-2010")
df_2010 = pd.read_excel("/content/online_retail_II.xlsx", sheet_name="Year 2010-2011")
df = pd.concat([df_2009, df_2010], ignore_index=True)

In [28]:
print(df.shape)
df.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


**1. Nettoyage: obtenir des transactions fiables (retirer annulations, retours, IDs manquants, valeurs aberrantes).**

In [29]:
#supprimer les valeurs manquantes
df.dropna(
    subset=['Customer ID'],
    inplace=True
)

**2. FEATURE RFM**

Cette méthode consiste à résumer le comportement d'achat de chaque client à l'aide de trois indicateurs:

*   La récence(R):Depuis combien de jours le client n'a-t-il rien acheté ? (Plus le chiffre est bas, mieux c'est).
*   Fréquence (F) : Combien de fois le client a-t-il acheté sur une période donnée ? (Plus le chiffre est haut, mieux c'est).
*   Montant (M) : Somme totale dépensée par ce client. (Plus le chiffre est haut, mieux c'est).

***Le résultat*** : Chaque client de la base se retrouve avec une fiche simple contenant uniquement 3 colonnes de chiffres : son score R, son score F et son score M.




In [30]:
#convertir la colonne Invoice Date au format datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

                            Calcul du montant total

**Calcul du montant total des ventes:**

***Montant total= Quantité X Prix unitaire***

In [31]:
#Calcul du montant total de ventes
df['montant_total'] = df['Quantity'] * df['Price']

In [32]:
df.head
df.tail()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,montant_total
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,12.60
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,16.60
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,16.60
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France,14.85
1067370,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.00,12680.0,France,18.00


             Détermination de la date de reférence

Nous devons savoir à quel moment les achats ont été effectués, et c’est pourquoi nous allons créer une date de référence:

***Date de reférence= La date maximale + 1 jour***

In [33]:
#Calcul de la date de reférence
from datetime import timedelta
reference_date = df['InvoiceDate'].max() + timedelta(days=1)

In [34]:
print(f"La date de reférence est {reference_date}")

La date de reférence est 2011-12-10 12:50:00


In [35]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,montant_total
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


**CALCUL DU RFM**

nous allons calculer les trois composantes du modèle RFM :


*   Recency (Récence) ;
*   Frequency (Fréquence) ;
*   Monetary Value (Valeur monétaire).




Nous allons regrouper les données par Custumer ID:


*   **InvoiceDate**: Pour chaque groupe de clients, x représente une série contenant toutes les dates de transaction de ce client.
*  **Invoice Number**: Pour chaque client, cette fonction compte le nombre d'entrées de factures
*   **Montant_total**: Pour chaque client, cela additionne toutes les valeurs de Total Amount et nous indique combien le client a dépensé chez nous.




In [36]:
RFM = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,
    'Invoice': 'count',
    'montant_total': 'sum'
})

Nous allons ensuite renommer InvoiceDate en Recence, Invoice number en fréquence puis montant_total en montant

In [37]:
RFM.rename(
    columns={
        'InvoiceDate': 'recence',
        'Invoice': 'frequence',
        'montant_total': 'montant'
    },
    inplace=True
)

In [38]:
RFM.head()

,recence,frequence,montant
Customer ID,,,
12346.0,326,48,-64.68
12347.0,2,253,5633.32
12348.0,75,51,2019.40
12349.0,19,180,4404.54
12350.0,310,17,334.40


L'étape suivante consiste à attribuer des scores à chacune des valeurs(Récence, Fréquence montant).
Pour cela, nous devons calculer les quantiles, ou percentiles, pour la récence, la fréquence et la valeur monétaire. Ainsi, cela permet de calculer, pour chaque colonne du DataFrame RFM :

le 25e percentile, c'est-à-dire le premier quartile ;
le 50e percentile, qui correspond à la médiane ;
le 75e percentile, qui correspond au troisième quartile.

In [39]:
#créer une nouvelle variable appelée quantiles
quantiles = RFM.quantile(
    q=[0.25, 0.5, 0.75]
)

Afin de nous faciliter le travail, nous allons créer une fonction scoring pour le calcul du score(recence, fréquence, Montant

In [40]:
def RFM_score(x, p, d):
    if p == 'recence':
        if x <= d[p][0.25]:
            return 4
        elif x <= d[p][0.50]:
            return 3
        elif x <= d[p][0.75]:
            return 2
        else:
            return 1
    else:
        if x <= d[p][0.25]:
            return 1
        elif x <= d[p][0.50]:
            return 2
        elif x <= d[p][0.75]:
            return 3
        else:
            return 4

Dans la généralisation de la fonction:


*   **x** représente la valeur à laquelle nous voulons attribuer un score. Il peut s'agir de la récence, de la fréquence ou de la valeur monétaire du client ;

*    **p** représente le nom de la colonne, donc recence, frequence ou montant ;

*   **d** représente le DataFrame contenant les quantiles.





In [41]:
RFM['R']= RFM['recence'].apply(RFM_score, args=('recence', quantiles))
RFM['F']= RFM['frequence'].apply(RFM_score, args=('frequence', quantiles))
RFM['M']= RFM['montant'].apply(RFM_score, args=('montant', quantiles))

In [42]:
RFM.head()

,recence,frequence,montant,R,F,M
Customer ID,,,,,,
12346.0,326,48,-64.68,2,2,1
12347.0,2,253,5633.32,4,4,4
12348.0,75,51,2019.40,3,2,3
12349.0,19,180,4404.54,4,4,4
12350.0,310,17,334.40,2,1,2


Nous allons créer une colonne appelée RFM Segment en convertissant les scores en chaînes de caractères.

Nous allons également calculer le score RFM, qui correspond au score total.

Nous allons additionner les scores de Récence, Fréquence et Valeur monétaire afin d'obtenir un score global.

In [43]:
RFM['RFM_segment'] = RFM['R'].astype(str) + RFM['F'].astype(str) + RFM['M'].astype(str)
RFM['RFM_score'] = RFM[['R', 'F', 'M']].sum(axis=1)

In [44]:
RFM.head()

,recence,frequence,montant,R,F,M,RFM_segment,RFM_score
Customer ID,,,,,,,,
12346.0,326,48,-64.68,2,2,1,221,5
12347.0,2,253,5633.32,4,4,4,444,12
12348.0,75,51,2019.40,3,2,3,323,8
12349.0,19,180,4404.54,4,4,4,444,12
12350.0,310,17,334.40,2,1,2,212,5
